# Listen to Pocket TTS training samples

Every `sample_freq` steps, training writes `<run_dir>/samples/step<N>_<i>.wav`: one file per sentence in `sample_sentences` of `args.yaml`.
If those sentences come from the validation split, section 2 plays the real recordings for comparison.

Put this notebook in `<run_dir>/samples/` (or set `RUN_DIR` below) and run all cells.

Notes:
- Samples use the raw (non-EMA) weights, CFG 2.0 and the **first training batch's voice prompt**, so the voice can change between steps. Judge words, rhythm and audio quality, not voice consistency.
- Re-run the notebook (*Run → Run All Cells*) to pick up new steps.

In [ ]:
from pathlib import Path
import json, re, wave
import yaml
from IPython.display import Audio, HTML, Markdown, display

RUN_DIR = None  # e.g. Path("../../runs/moore_fr24l"); None = this notebook sits in <run_dir>/samples
RUN_DIR = Path(RUN_DIR) if RUN_DIR else Path.cwd().parent
SAMPLES_DIR = RUN_DIR / "samples"
assert (RUN_DIR / "args.yaml").exists(), f"no args.yaml in {RUN_DIR}: set RUN_DIR"

args = yaml.safe_load((RUN_DIR / "args.yaml").read_text())
SENTENCES = args["sample_sentences"]
VALID_JSONL = Path(args["data"]["valid_jsonl"])
if not VALID_JSONL.is_absolute():  # configs use paths relative to the pocket-tts checkout
    VALID_JSONL = (RUN_DIR / ".." / ".." / "pocket-tts" / VALID_JSONL).resolve()

def wav_seconds(path):
    with wave.open(str(path)) as w:
        return w.getnframes() / w.getframerate()

samples = {}  # step -> {sentence index: path}
for p in SAMPLES_DIR.glob("step*_*.wav"):
    m = re.fullmatch(r"step(\d+)_(\d+)\.wav", p.name)
    if m:
        samples.setdefault(int(m[1]), {})[int(m[2])] = p
STEPS = sorted(samples)
print(f"run: {RUN_DIR.name} | {len(STEPS)} sample rounds: {STEPS}")

## 1. Latest step

In [ ]:
step = STEPS[-1]  # or pick one, e.g. step = 15000
display(Markdown(f"### Step {step:,}"))
for i, text in enumerate(SENTENCES):
    path = samples[step].get(i)
    display(Markdown(f"**{i}.** {text}"))
    if path is None:
        print("  (missing)")
        continue
    print(f"  {wav_seconds(path):.1f} s")
    display(Audio(filename=str(path)))

## 2. Real recordings of the same sentences (held-out chapters)

The sentences were taken from validation segments, which can contain a few neighbouring verses too. This is the reference for pronunciation and pacing.

In [ ]:
refs = [json.loads(line) for line in open(VALID_JSONL)]
for i, text in enumerate(SENTENCES):
    match = next((r for r in refs if text in r["transcript"]), None)
    display(Markdown(f"**{i}.** {text}"))
    if match is None:
        print("  (no validation segment contains this sentence)")
        continue
    print(f"  {match['speaker']} | {match['chapter']} | {match['duration']:.1f} s | full segment text: {match['transcript']}")
    display(Audio(filename=match["path"]))

## 3. Progress: the same sentence across steps

Set `SENTENCE` and `SHOW_STEPS`. Each audio player is embedded in the notebook, so keep `SHOW_STEPS` short (the notebook grows by ~0.3 MB per clip).

In [ ]:
SENTENCE = 0  # index into SENTENCES
SHOW_STEPS = STEPS[:1] + STEPS[-4:] if len(STEPS) > 5 else STEPS  # first + last 4; or e.g. [2500, 10000, 15000]

display(Markdown(f"**{SENTENCE}.** {SENTENCES[SENTENCE]}"))
for step in dict.fromkeys(SHOW_STEPS):
    path = samples.get(step, {}).get(SENTENCE)
    if path is None:
        print(f"step {step:>7,}: (missing)")
        continue
    print(f"step {step:>7,}: {wav_seconds(path):.1f} s")
    display(Audio(filename=str(path)))

## 4. Loss curves (`progress.jsonl`)

- Teacher runs: **valid `flow_diag`** is the plain reconstruction error (lower is better; should fall, then flatten). `loss` includes a learned uncertainty term and can be negative.
- Distillation runs: **valid `distill_mse`**, the student's error against the teacher (lower is better).
- If validation rises while training keeps falling, the model is overfitting: keep an earlier checkpoint (only the last 3 are kept).

In [ ]:
import matplotlib.pyplot as plt

train, valid = [], []
for line in open(RUN_DIR / "progress.jsonl"):
    e = json.loads(line)
    if e.get("type") == "train":
        train.append(e)
    elif e.get("type") == "valid":
        valid.append(e)

keys = [k for k in ("flow_diag", "distill_mse", "loss") if valid and k in valid[-1]["metrics"]] or ["loss"]
print(f"{'step':>8} | " + " | ".join(f"valid {k:>11}" for k in keys))
for e in valid:
    print(f"{e['step']:>8,} | " + " | ".join(f"{float(e['metrics'].get(k, 'nan')):>17.4f}" for k in keys))

def smooth(values, k=20):
    out, acc = [], []
    for v in values:
        acc.append(v)
        acc = acc[-k:]
        out.append(sum(acc) / len(acc))
    return out

fig, axes = plt.subplots(1, len(keys), figsize=(6 * len(keys), 4), squeeze=False)
for ax, key in zip(axes[0], keys):
    tr = [e for e in train if key in e["metrics"]]
    ax.plot([e["step"] for e in tr], smooth([float(e["metrics"][key]) for e in tr]), label="train (smoothed)")
    ax.plot([e["step"] for e in valid], [float(e["metrics"][key]) for e in valid], "o-", label="valid")
    ax.set_title(key)
    ax.set_xlabel("step")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()